# Recipe Chatbot with Memory — Chef Orbit

**Lumexa AI Builder course**

A conversational recipe assistant, **Chef Orbit**, that remembers dietary
restrictions and preferences you mention mid-conversation — and uses that
memory to filter its recipe suggestions.

This notebook is fully self-contained and works with **Runtime → Run all** —
no setup, no API keys required, and nothing to upload.

**What you'll do:**
1. Build Chef Orbit's persona and a `conversation_history` list — the core
   memory mechanism (the same idea from the original command-line version of
   this project).
2. Add an explicit, inspectable "remembered facts" dictionary that tracks
   dietary restrictions detected in what you say.
3. Chat with Chef Orbit through a `chat()` function and watch a scripted
   demo conversation prove that memory actually changes its suggestions.
4. Learn about an *optional* upgrade path: paste in a real OpenAI API key to
   let Chef Orbit reply using `gpt-4o-mini` instead of the built-in
   rule-based engine — completely optional, and the notebook works perfectly
   without it.

**Why no required API key / no downloaded AI model?** This project runs
anywhere for free, instantly, with zero setup friction — that's the whole
point of the "default path". If you want to see a real large language model
in action, the optional OpenAI section at the bottom lets you try that too.


## Step 1: Chef Orbit's persona

This is the same persona and purpose text from the original
`src/chatbot.py` command-line chatbot. It's used two ways in this notebook:

- As the literal system prompt if you opt in to the real OpenAI API later.
- As the "flavor" and boundaries that our own rule-based reply engine
  follows (light space metaphors, staying on-topic, being encouraging).


In [1]:
SYSTEM_PROMPT = """
You are Chef Orbit, a warm and encouraging AI recipe assistant aboard the
Lumexa space station galley.

PERSONA:
- You speak in a friendly, enthusiastic tone, like an experienced home cook
  who loves helping people in the kitchen.
- You occasionally use light space/cooking metaphors ("let's launch into
  this recipe"), but at most one per response.
- Keep responses focused and practical: ingredients, steps, or substitutions,
  not long unrelated stories.

PURPOSE:
- Help the user find recipes, suggest ingredient substitutions, and answer
  cooking questions.
- Pay close attention to any dietary restrictions, allergies, or preferences
  the user mentions at ANY point in the conversation (e.g., "I'm vegetarian",
  "I'm allergic to peanuts", "I don't eat dairy"). Remember these for the
  rest of the conversation and never suggest a recipe or ingredient that
  conflicts with them.

BOUNDARIES:
- If asked something unrelated to food, cooking, or nutrition, gently
  redirect back to the kitchen: "That's outside my galley duties — let's
  get back to cooking!"
- If you don't know something (like a very obscure ingredient), say so
  honestly rather than inventing an answer.

Stay in character as Chef Orbit for the entire conversation.
""".strip()

print(SYSTEM_PROMPT[:120] + "...")


You are Chef Orbit, a warm and encouraging AI recipe assistant aboard the
Lumexa space station galley.

PERSONA:
- You s...


## Step 2: Conversation memory — the growing history list

This list **is** Chef Orbit's memory. Every user message and every assistant
reply gets appended here, and (in the optional OpenAI mode) the *entire*
list is resent on every API call — that's what lets an LLM "remember"
something you said several turns ago.

We also keep a separate, explicit `remembered_facts` dictionary. Rather than
hoping a language model informally recalls "the user is vegetarian" from a
wall of text, we **detect it with simple keyword matching and store it as
real, inspectable Python state** — arguably a more reliable and more
teachable form of memory for a small assistant like this one.


In [2]:
conversation_history = [
    {"role": "system", "content": SYSTEM_PROMPT},
]

remembered_facts = {
    "dietary_restrictions": set(),
}

print("conversation_history has", len(conversation_history), "message(s)")
print("remembered_facts:", remembered_facts)


conversation_history has 1 message(s)
remembered_facts: {'dietary_restrictions': set()}


## Step 3: Detecting dietary facts from what you say

`extract_facts()` scans each user message for simple phrases
("vegetarian", "allergic to peanuts", "no dairy", ...) and adds any matches
to `remembered_facts["dietary_restrictions"]`. This runs on *every* user
message, so a restriction mentioned in turn 1 is still remembered in turn 5.


In [3]:
RESTRICTION_PATTERNS = {
    "vegetarian": ["vegetarian"],
    "vegan": ["vegan"],
    "pescatarian": ["pescatarian"],
    "gluten-free": ["gluten-free", "gluten free", "celiac", "coeliac"],
    "dairy-free": ["dairy-free", "dairy free", "lactose intolerant", "no dairy", "don't eat dairy", "not eat dairy"],
    "nut allergy": ["allergic to nuts", "nut allergy", "peanut allergy", "allergic to peanuts", "tree nut"],
    "halal": ["halal"],
    "kosher": ["kosher"],
}


def extract_facts(user_message, facts):
    """Scan a user message for dietary-restriction keywords and remember them."""
    lowered = user_message.lower()
    for label, patterns in RESTRICTION_PATTERNS.items():
        if any(pattern in lowered for pattern in patterns):
            facts["dietary_restrictions"].add(label)
    return facts


# Quick sanity check
test_facts = {"dietary_restrictions": set()}
extract_facts("Hi! I'm vegetarian and also allergic to peanuts.", test_facts)
print(test_facts)


{'dietary_restrictions': {'vegetarian', 'nut allergy'}}


## Step 4: A small recipe bank, filtered by remembered restrictions

Each recipe records what it `contains` (meat, fish, dairy, egg, gluten,
nuts, pork). A recipe is only suggested if none of the user's remembered
restrictions conflict with what it contains — this is the concrete
mechanism that makes "memory actually changes future suggestions" true,
not just a hope that an LLM behaves itself.


In [4]:
RESTRICTION_EXCLUDES = {
    "vegetarian": {"meat", "fish"},
    "vegan": {"meat", "fish", "dairy", "egg"},
    "pescatarian": {"meat"},
    "gluten-free": {"gluten"},
    "dairy-free": {"dairy"},
    "nut allergy": {"nuts"},
    "halal": {"pork"},
    "kosher": {"pork"},
}

RECIPES = [
    {"name": "rainbow veggie stir-fry with crispy tofu", "meal": "dinner", "contains": set()},
    {"name": "hearty chickpea and spinach curry with rice", "meal": "dinner", "contains": set()},
    {"name": "creamy mushroom risotto", "meal": "dinner", "contains": {"dairy"}},
    {"name": "grilled lemon-herb salmon with roasted vegetables", "meal": "dinner", "contains": {"fish"}},
    {"name": "classic beef and bean tacos", "meal": "dinner", "contains": {"meat"}},
    {"name": "peanut-ginger noodle bowl", "meal": "dinner", "contains": {"nuts", "gluten"}},
    {"name": "loaded black bean quesadillas", "meal": "dinner", "contains": {"dairy", "gluten"}},
    {"name": "honey-glazed pork stir-fry", "meal": "dinner", "contains": {"meat", "pork"}},
    {"name": "overnight oats with berries and almond butter", "meal": "breakfast", "contains": {"nuts"}},
    {"name": "fluffy scrambled eggs with avocado toast", "meal": "breakfast", "contains": {"egg", "gluten"}},
    {"name": "tropical fruit and coconut chia pudding", "meal": "breakfast", "contains": set()},
    {"name": "veggie and cheese omelet", "meal": "breakfast", "contains": {"egg", "dairy"}},
    {"name": "rustic lentil soup with crusty bread", "meal": "lunch", "contains": {"gluten"}},
    {"name": "grilled chicken caesar salad", "meal": "lunch", "contains": {"meat", "dairy", "gluten"}},
    {"name": "quinoa tabbouleh with roasted chickpeas", "meal": "lunch", "contains": set()},
    {"name": "tuna salad lettuce wraps", "meal": "lunch", "contains": {"fish"}},
    {"name": "dark chocolate avocado mousse", "meal": "dessert", "contains": set()},
    {"name": "classic New York cheesecake", "meal": "dessert", "contains": {"dairy", "egg", "gluten"}},
    {"name": "fresh fruit salad with mint and lime", "meal": "dessert", "contains": set()},
    {"name": "almond flour lemon cake", "meal": "dessert", "contains": {"nuts", "egg"}},
    {"name": "peanut butter chocolate chip cookies", "meal": "dessert", "contains": {"nuts", "gluten", "dairy", "egg"}},
]


def recipe_is_compatible(recipe, restrictions):
    for restriction in restrictions:
        excluded = RESTRICTION_EXCLUDES.get(restriction, set())
        if recipe["contains"] & excluded:
            return False
    return True


print(f"{len(RECIPES)} recipes loaded.")


21 recipes loaded.


## Step 5: The rule-based reply engine (default path, no key needed)

`generate_rule_based_reply()` figures out roughly what you're asking for
(dinner idea? dessert? a substitution? just saying hi?), filters the recipe
bank by your remembered restrictions, and phrases a reply in Chef Orbit's
voice using a few randomly-picked template variants so it doesn't sound
robotic and identical every time.


In [5]:
import random
import re

RNG_SEED = 7
rng = random.Random(RNG_SEED)


def has_phrase(text, phrase):
    """Whole-word/phrase match so short words like 'eat' don't match inside
    unrelated words like 'weather' (a real bug we caught while testing!)."""
    pattern = r"\b" + re.escape(phrase) + r"\b"
    return re.search(pattern, text) is not None


MEAL_KEYWORDS = {
    "dessert": ["dessert", "sweet", "cake", "cookie", "treat"],
    "breakfast": ["breakfast", "morning meal"],
    "lunch": ["lunch", "midday meal"],
    "dinner": ["dinner", "supper", "main course", "entree", "tonight"],
}
GREETING_KEYWORDS = ["hi", "hello", "hey", "greetings"]
SUBSTITUTION_KEYWORDS = ["substitute", "instead of", "replace", "swap", "alternative"]
OFF_TOPIC_KEYWORDS = ["weather", "homework", "math problem", "stock market", "president", "election", "sports score"]
FOOD_HINT_KEYWORDS = ["recipe", "cook", "eat", "meal", "food", "dish", "ingredient"]

INTRO_PHRASES = [
    "Let's launch into this recipe: ",
    "Here's a tasty idea for you: ",
    "Great choice, cadet! How about this: ",
    "Straight from the galley: ",
]
RESTRICTION_ACK_TEMPLATES = [
    "Since you mentioned you're {restrictions}, here's a suggestion that fits perfectly: ",
    "Keeping your {restrictions} needs in mind, try this: ",
    "No problem cooking around {restrictions} — here's one for you: ",
]
NO_MATCH_TEMPLATES = [
    "Hmm, my current recipe bank doesn't have a {meal} option that fits {restrictions} — "
    "could you tell me a bit more about what you're craving so I can improvise?",
]
SUBSTITUTION_REPLY = (
    "Good question! A safe general swap is: unsweetened applesauce or mashed banana for "
    "eggs in baking, coconut or oat milk for dairy milk, and gluten-free flour blends for "
    "wheat flour. Tell me the exact recipe and I can get more specific!"
)
GREETING_REPLIES = [
    "Hello, cadet! What are we cooking today?",
    "Hey there! Ready to launch into some cooking?",
    "Greetings from the galley! What can I help you cook?",
]
BOUNDARY_REPLY = "That's outside my galley duties — let's get back to cooking!"


def detect_meal_type(lowered_message):
    for meal, keywords in MEAL_KEYWORDS.items():
        if any(has_phrase(lowered_message, keyword) for keyword in keywords):
            return meal
    return None


def generate_rule_based_reply(user_message, facts, rng):
    lowered = user_message.lower()
    restrictions = sorted(facts["dietary_restrictions"])
    restriction_text = " and ".join(restrictions) if restrictions else ""

    if any(has_phrase(lowered, keyword) for keyword in GREETING_KEYWORDS) and len(lowered.split()) < 6:
        return rng.choice(GREETING_REPLIES)

    if any(has_phrase(lowered, keyword) for keyword in OFF_TOPIC_KEYWORDS) and not any(
        has_phrase(lowered, keyword) for keyword in FOOD_HINT_KEYWORDS
    ):
        return BOUNDARY_REPLY

    if any(has_phrase(lowered, keyword) for keyword in SUBSTITUTION_KEYWORDS):
        return SUBSTITUTION_REPLY

    meal = detect_meal_type(lowered) or "dinner"
    candidates = [r for r in RECIPES if r["meal"] == meal and recipe_is_compatible(r, restrictions)]

    if not candidates:
        return rng.choice(NO_MATCH_TEMPLATES).format(meal=meal, restrictions=restriction_text or "your preferences")

    choice = rng.choice(candidates)
    if restrictions:
        opener = rng.choice(RESTRICTION_ACK_TEMPLATES).format(restrictions=restriction_text)
    else:
        opener = rng.choice(INTRO_PHRASES)

    return f"{opener}{choice['name']}. Let me know if you'd like the step-by-step recipe!"


# Quick manual check (not part of the graded demo)
print(generate_rule_based_reply("What should I make for dinner?", {"dietary_restrictions": {"vegetarian"}}, rng))


Since you mentioned you're vegetarian, here's a suggestion that fits perfectly: creamy mushroom risotto. Let me know if you'd like the step-by-step recipe!


## Step 6: Optional upgrade — a real OpenAI API key

Leave the field below **blank** to keep using the free rule-based engine
above (this is the default, and it's what makes `Runtime → Run all` always
succeed with no setup). If you paste in a real key, Chef Orbit will try to
reply using `gpt-4o-mini` for a noticeably more natural conversation — and
if that call fails for *any* reason (bad key, no quota, no internet), the
notebook quietly falls back to the rule-based engine instead of crashing.


In [6]:
%pip install -q openai
print("openai package ready (only used if you provide an API key below).")


Note: you may need to restart the kernel to use updated packages.
openai package ready (only used if you provide an API key below).


In [7]:
OPENAI_API_KEY = ""  #@param {type:"string"}


In [8]:
def try_openai_reply(history):
    """Attempt a real OpenAI call. Returns the reply text, or None on any failure."""
    if not OPENAI_API_KEY:
        return None
    try:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=history,
            temperature=0.7,
            max_tokens=350,
        )
        return response.choices[0].message.content
    except Exception as error:
        print(f"(OpenAI call failed, falling back to the local engine: {error})")
        return None


## Step 7: Putting it together — `chat()` and `print_history()`

`chat(user_message)` is the function you'll call directly. Each call:

1. Appends your message to `conversation_history`.
2. Updates `remembered_facts` from anything you said.
3. Tries the optional OpenAI path, falling back to the rule-based engine.
4. Appends the reply to `conversation_history` and prints the exchange.

`print_history()` is the equivalent of the original CLI's `/history`
debug command — it prints the raw memory list being tracked.


In [9]:
def chat(user_message):
    conversation_history.append({"role": "user", "content": user_message})
    extract_facts(user_message, remembered_facts)

    reply = try_openai_reply(conversation_history)
    if reply is None:
        reply = generate_rule_based_reply(user_message, remembered_facts, rng)

    conversation_history.append({"role": "assistant", "content": reply})

    print(f"You: {user_message}")
    print(f"Chef Orbit: {reply}\n")
    return reply


def print_history():
    """Debug helper: print the raw conversation memory list (like the CLI's /history)."""
    print("--- Raw conversation history ---")
    for i, message in enumerate(conversation_history):
        print(f"[{i}] {message['role'].upper()}: {message['content']}")
    print("--- end of history ---")
    print("Remembered facts:", remembered_facts)


## Step 8: Scripted demo — memory in action

This runs automatically with no typing required, so `Runtime → Run all`
always completes. Watch how the dinner suggestion in turn 2 already respects
the vegetarian restriction mentioned in turn 1, and how the dessert
suggestion in turn 3 respects it too.


In [10]:
chat("Hi Chef Orbit! I'm vegetarian.")
chat("What should I make for dinner tonight?")
chat("Any dessert ideas?")
chat("By the way, what's the weather like today?")


You: Hi Chef Orbit! I'm vegetarian.
Chef Orbit: Hey there! Ready to launch into some cooking?

You: What should I make for dinner tonight?
Chef Orbit: Since you mentioned you're vegetarian, here's a suggestion that fits perfectly: rainbow veggie stir-fry with crispy tofu. Let me know if you'd like the step-by-step recipe!

You: Any dessert ideas?
Chef Orbit: Since you mentioned you're vegetarian, here's a suggestion that fits perfectly: peanut butter chocolate chip cookies. Let me know if you'd like the step-by-step recipe!

You: By the way, what's the weather like today?
Chef Orbit: That's outside my galley duties — let's get back to cooking!



"That's outside my galley duties — let's get back to cooking!"

**Honest note on the rule-based engine:** this default, no-key-required
engine only recognizes the specific dietary phrases it's been taught to
look for (see `RESTRICTION_PATTERNS` above), and its replies are built from
a fixed bank of recipes and phrasing templates rather than truly generated
text — so it can't hold an open-ended conversation the way `gpt-4o-mini`
can. What it *can* do reliably is guarantee that once a restriction is
detected, it will never be violated by a later suggestion — you can verify
that yourself by reading the recipe/contains/exclude logic above. If you
add a real OpenAI key in Step 6, re-run the demo cell and compare: the
replies become far more natural and flexible, though (being a general
LLM) it is asked, not guaranteed, to honor the same restrictions.


In [11]:
print_history()


--- Raw conversation history ---
[0] SYSTEM: You are Chef Orbit, a warm and encouraging AI recipe assistant aboard the
Lumexa space station galley.

PERSONA:
- You speak in a friendly, enthusiastic tone, like an experienced home cook
  who loves helping people in the kitchen.
- You occasionally use light space/cooking metaphors ("let's launch into
  this recipe"), but at most one per response.
- Keep responses focused and practical: ingredients, steps, or substitutions,
  not long unrelated stories.

PURPOSE:
- Help the user find recipes, suggest ingredient substitutions, and answer
  cooking questions.
- Pay close attention to any dietary restrictions, allergies, or preferences
  the user mentions at ANY point in the conversation (e.g., "I'm vegetarian",
  "I'm allergic to peanuts", "I don't eat dairy"). Remember these for the
  rest of the conversation and never suggest a recipe or ingredient that
  conflicts with them.

BOUNDARIES:
- If asked something unrelated to food, cooking, or

## Step 9: Try it yourself (optional)

Edit the message below and re-run this cell as many times as you like — it
keeps adding to the same `conversation_history`, so Chef Orbit keeps
remembering everything you've said so far in this notebook session.


In [12]:
chat("Your message here")


You: Your message here
Chef Orbit: No problem cooking around vegetarian — here's one for you: creamy mushroom risotto. Let me know if you'd like the step-by-step recipe!



"No problem cooking around vegetarian — here's one for you: creamy mushroom risotto. Let me know if you'd like the step-by-step recipe!"